# Mouse CNS Single Sample Tutorial (Human PPI Gene Programs)

- **Creator**: Sebastian Birk (<sebastian.birk@helmholtz-munich.de>).
- **Affiliation:** Helmholtz Munich, Institute of AI for Health (AIH), Talavera-López Lab
- **Date of Creation:** 16.09.2026
- **Date of Last Modification:** 16.09.2026

This tutorial trains NicheCompass on the same data as the
[single sample tutorial](mouse_cns_single_sample.ipynb) -- one sagittal brain
section of the STARmap PLUS mouse central nervous system dataset from
[Shi, H. et al. Nature 622, 552-561 (2023)](https://www.nature.com/articles/s41586-023-06569-5)
-- but builds the prior gene program mask from **one resource only**: the
predicted human interactome of
[Zhang, J., Humphreys, I. R. et al. Science (2025)](https://doi.org/10.1126/science.adt1630).

The sample has:
- 91,246 observations at cellular resolution with cell type annotations
- 1022 probed genes

**Why a separate tutorial.** The default mask combines OmniPath, NicheNet and
MEBOCOST, all of which are curated ligand-receptor or enzyme-sensor resources
that already state a direction. The human PPI resource does not: it is an
undirected list of predicted protein pairs, dominated by intracellular
complexes, so it has to be classified into signalling directions before it can
serve as a neighbour-to-self prior. Using it alone makes that classification
visible, and makes it obvious which programs survive into the mask. The
[human PPI guide](../../user_guide/humanppi_gene_programs.md) documents the
classification step by step.

**What to expect.** A single-resource mask is much smaller than the default
one, and this panel measures only 1022 genes, so most predicted interactions
have no measured partner and drop out. Expect tens of prior GPs rather than
hundreds, and read the count the notebook prints before interpreting anything.
This is a demonstration of the resource, not a recommendation to train on it
alone.

- Check the [documentation](https://nichecompass.readthedocs.io/en/latest/installation.html) for NicheCompass installation instructions.
- The data for this tutorial can be downloaded from [Google Drive](https://drive.google.com/drive/folders/1l9W0MDVZ451k1L7s6GGH4ONH4tEK4EKj). It has to be stored under ```<repository_root>/data/spatial_omics/```.
  - starmap_plus_mouse_cns_batch1.h5ad
- The human PPI predictions are downloaded on first use from
  [conglab.swmed.edu/humanPPI](https://conglab.swmed.edu/humanPPI/) and cached
  under ```<repository_root>/data/gene_programs/```. On a machine without
  internet access, populate that folder first; the cache layout is
  documented under "Retrieve the prior gene program caches first" in the
  [multi-GPU guide](../../user_guide/multi_gpu_training.md).

## 1. Setup

### 1.1 Import Libraries

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import warnings
from datetime import datetime

import scanpy as sc
import squidpy as sq

from nichecompass.models import NicheCompass
from nichecompass.utils import (add_gps_from_gp_dict_to_adata,
                                extract_gp_dict_from_humanppi_interactions,
                                filter_and_combine_gp_dict_gps_v2,
                                get_unique_genes_from_gp_dict)

### 1.2 Define Parameters

In [ ]:
### Dataset ###
dataset = "starmap_plus_mouse_cns"
species = "mouse"
spatial_key = "spatial"
n_neighbors = 4

### Prior GPs: human PPI only ###
# ´precision´ selects the published confidence tier: "90" gives 17,849
# interactions at 90% expected precision, "80" gives 29,257 at 80%.
humanppi_precision = "90"
# ´intercellular´ keeps only the programs that can act between a cell and its
# neighbours, which is what NicheCompass reconstructs. "intracellular" and
# "both" are available but a neighbourhood model cannot use an intracellular
# complex as a spatial prior.
humanppi_program_type = "intercellular"
# A partner whose annotation is compatible with an extracellular face without
# establishing one (a bare ´Membrane´ keyword, say). "extracellular" keeps it,
# "intracellular" discards it. This is the single most consequential choice
# for how many programs survive.
humanppi_ambiguous_locality = "extracellular"
# A partner with no usable location annotation at all.
humanppi_unresolved_locality = "exclude"
# Use predicted protein topology to decide which partner reaches across the
# junction, and reclassify a contact-dependent pair as a cis complex when the
# shorter partner does not protrude far enough to reach a neighbour.
humanppi_use_topology = True
humanppi_detect_cis_complexes = True
humanppi_min_extracellular_domain_length = 30
# Orient juxtacrine programs so the sending and receiving arms are distinct,
# rather than putting both partners in both arms.
humanppi_orient_juxtacrine_gps = True
humanppi_symmetric_juxtacrine_gps = False

### Model ###
# AnnData Keys
counts_key = "counts"
adj_key = "spatial_connectivities"
gp_names_key = "nichecompass_gp_names"
active_gp_names_key = "nichecompass_active_gp_names"
gp_targets_mask_key = "nichecompass_gp_targets"
gp_targets_categories_mask_key = "nichecompass_gp_targets_categories"
gp_sources_mask_key = "nichecompass_gp_sources"
gp_sources_categories_mask_key = "nichecompass_gp_sources_categories"
latent_key = "nichecompass_latent"

# Architecture
conv_layer_encoder = "gcnconv" # change to "gatv2conv" if enough compute and memory
active_gp_thresh_ratio = 0.01

# Trainer
n_epochs = 400
n_epochs_all_gps = 25
lr = 0.001
lambda_edge_recon = 500000.
lambda_gene_expr_recon = 300.
lambda_l1_masked = 0. # prior GP regularization
lambda_l1_addon = 30. # de novo GP regularization
edge_batch_size = 1024 # increase if more memory available or decrease to save memory
n_sampled_neighbors = 4
use_cuda_if_available = True

### Analysis ###
cell_type_key = "Main_molecular_cell_type"
latent_leiden_resolution = 0.4
latent_cluster_key = f"latent_leiden_{str(latent_leiden_resolution)}"
sample_key = "batch"
spot_size = 0.2

### 1.3 Run Notebook Setup

In [ ]:
warnings.filterwarnings("ignore")

In [ ]:
# Get time of notebook execution for timestamping saved artifacts
now = datetime.now()
current_timestamp = now.strftime("%d%m%Y_%H%M%S")

### 1.4 Configure Paths

In [ ]:
# Define paths
ga_data_folder_path = "../../../data/gene_annotations"
gp_data_folder_path = "../../../data/gene_programs"
so_data_folder_path = "../../../data/spatial_omics"
gene_orthologs_mapping_file_path = f"{ga_data_folder_path}/human_mouse_gene_orthologs.csv"
# Caches for the human PPI workflow. Each is downloaded on first use and
# reused afterwards; the network file name carries the precision tier because
# the two tiers are different downloads.
humanppi_network_file_path = f"{gp_data_folder_path}/humanppi_network_{humanppi_precision}.csv"
humanppi_topology_file_path = f"{gp_data_folder_path}/humanppi_protein_topology.tsv"
omnipath_annotation_file_path = f"{gp_data_folder_path}/omnipath_intercell_annotation.tsv"
complex_portal_file_path = f"{gp_data_folder_path}/complex_portal_human.tsv"
artifacts_folder_path = f"../../../artifacts"
model_folder_path = f"{artifacts_folder_path}/single_sample_humanppi/{current_timestamp}/model"
figure_folder_path = f"{artifacts_folder_path}/single_sample_humanppi/{current_timestamp}/figures"


### 1.5 Create Directories

In [ ]:
os.makedirs(model_folder_path, exist_ok=True)
os.makedirs(figure_folder_path, exist_ok=True)
os.makedirs(so_data_folder_path, exist_ok=True)
os.makedirs(gp_data_folder_path, exist_ok=True)

## 2. Prepare Model Training

### 2.1 Create the Human PPI Gene Program Mask

The resource is an undirected edge list of predicted protein pairs, so the
extractor has to decide, for every pair, whether it could act between two cells
at all and which partner sends. It does that in three steps:

1. **Locate both partners** from their UniProt cellular-component keywords, as
   `cell_surface`, `secreted`, `intracellular` or `ambiguous`.
2. **Classify the pair** from that combination -- secreted-to-surface becomes a
   paracrine program, surface-to-surface a juxtacrine one, and anything with no
   extracellular face is either dropped or kept as an intracellular complex,
   depending on `humanppi_program_type`.
3. **Check reach** using predicted topology: a contact-dependent pair whose
   shorter partner does not protrude `min_extracellular_domain_length`
   residues from the membrane cannot reach a neighbouring cell, so it is
   reclassified as a cis complex within one membrane.

Because the resource is human and this dataset is mouse, `species="mouse"`
additionally maps every gene through the human-mouse ortholog table. Genes with
no one-to-one ortholog are lost at that step.

In [ ]:
# Retrieve human PPI GPs. The first call downloads and caches the
# predictions, the protein topology table, the OmniPath annotations and the
# Complex Portal reference; later calls reuse them from disk.
humanppi_load_from_disk = os.path.exists(humanppi_network_file_path)

humanppi_gp_dict = extract_gp_dict_from_humanppi_interactions(
    species=species,
    precision=humanppi_precision,
    program_type=humanppi_program_type,
    ambiguous_locality=humanppi_ambiguous_locality,
    unresolved_locality=humanppi_unresolved_locality,
    use_topology=humanppi_use_topology,
    topology_file_path=humanppi_topology_file_path,
    detect_cis_complexes=humanppi_detect_cis_complexes,
    min_extracellular_domain_length=humanppi_min_extracellular_domain_length,
    orient_juxtacrine_gps=humanppi_orient_juxtacrine_gps,
    omnipath_annotation_file_path=omnipath_annotation_file_path,
    symmetric_juxtacrine_gps=humanppi_symmetric_juxtacrine_gps,
    complex_portal_file_path=complex_portal_file_path,
    load_from_disk=humanppi_load_from_disk,
    save_to_disk=not humanppi_load_from_disk,
    ppi_network_file_path=humanppi_network_file_path,
    gene_orthologs_mapping_file_path=gene_orthologs_mapping_file_path,
    plot_gp_gene_count_distributions=True,
    gp_gene_count_distributions_save_path=f"{figure_folder_path}"
                                          "/humanppi_gp_gene_count_distributions.svg")

print(f"Number of human PPI gene programs: {len(humanppi_gp_dict)}.")

Programs are named after the interaction class they were assigned to, so the breakdown of the mask is readable straight off the names.

In [ ]:
# Breakdown by interaction class
from collections import Counter

classes = Counter(name.rsplit("_", 2)[-2] if name.count("_") >= 2 else name
                  for name in humanppi_gp_dict)
for gp_class, count in classes.most_common():
    print(f"{gp_class:>20}: {count}")

In [ ]:
# Display an example human PPI GP
example_gp = next(iter(humanppi_gp_dict))
print(f"{example_gp}:")
for entity, members in humanppi_gp_dict[example_gp].items():
    print(f"  {entity}: {members}")

In [ ]:
# Filter and combine GPs. With a single resource this mostly removes
# duplicates and programs that are subsets of another, rather than merging
# across resources as it does for the default mask.
gp_dicts = [humanppi_gp_dict]
combined_gp_dict = filter_and_combine_gp_dict_gps_v2(
    gp_dicts,
    verbose=True)

print(f"Number of gene programs after filtering and combining: "
      f"{len(combined_gp_dict)}.")

### 2.2 Load Data & Compute Spatial Neighbor Graph

- NicheCompass expects a precomputed spatial adjacency matrix stored in 'adata.obsp[adj_key]'.
- The user can customize the spatial neighbor graph construction based on the biological question of interest.

In [ ]:
# Read data
adata = sc.read_h5ad(
        f"{so_data_folder_path}/{dataset}_batch1.h5ad")

In [ ]:
# Compute spatial neighborhood
sq.gr.spatial_neighbors(adata,
                        coord_type="generic",
                        spatial_key=spatial_key,
                        n_neighs=n_neighbors)

# Make adjacency matrix symmetric
adata.obsp[adj_key] = (
    adata.obsp[adj_key].maximum(
        adata.obsp[adj_key].T))

### 2.3 Check Coverage Against the Measured Panel

This is the step that decides whether a single-resource mask is usable at
all. A program only enters the mask if at least one source gene and one target
gene were actually measured, so a 1022-gene panel discards most of a
proteome-scale resource. Look at the numbers below before going further: if
only a handful of programs survive, the latent space will be dominated by the
de novo programs and the prior part will carry little signal.

In [ ]:
gp_genes = get_unique_genes_from_gp_dict(
    gp_dict=combined_gp_dict,
    retrieved_gene_entities=["sources", "targets"])
measured = set(adata.var_names)
overlap = measured.intersection(gp_genes)

print(f"Genes in the human PPI mask       : {len(gp_genes)}")
print(f"Genes measured in this panel      : {len(measured)}")
print(f"Overlap                           : {len(overlap)}")
print(f"Share of the mask that is measured: {100 * len(overlap) / len(gp_genes):.1f}%")

### 2.4 Add GP Mask to Data

In [ ]:
# Add the GP dictionary as binary masks to the adata
add_gps_from_gp_dict_to_adata(
    gp_dict=combined_gp_dict,
    adata=adata,
    gp_targets_mask_key=gp_targets_mask_key,
    gp_targets_categories_mask_key=gp_targets_categories_mask_key,
    gp_sources_mask_key=gp_sources_mask_key,
    gp_sources_categories_mask_key=gp_sources_categories_mask_key,
    gp_names_key=gp_names_key,
    min_genes_per_gp=2,
    min_source_genes_per_gp=1,
    min_target_genes_per_gp=1,
    max_genes_per_gp=None,
    max_source_genes_per_gp=None,
    max_target_genes_per_gp=None)

print(f"Prior gene programs retained in the mask: "
      f"{len(adata.uns[gp_names_key])}.")

If that last number is small, say under about twenty, treat the run below as
a demonstration of the resource rather than as an analysis. The
[single sample tutorial](mouse_cns_single_sample.ipynb) combines three
resources and is the right starting point for real work on this panel.

## 3. Train Model

### 3.1 Initialize, Train & Save Model

In [ ]:
# Initialize model
model = NicheCompass(adata,
                     counts_key=counts_key,
                     adj_key=adj_key,
                     gp_names_key=gp_names_key,
                     active_gp_names_key=active_gp_names_key,
                     gp_targets_mask_key=gp_targets_mask_key,
                     gp_targets_categories_mask_key=gp_targets_categories_mask_key,
                     gp_sources_mask_key=gp_sources_mask_key,
                     gp_sources_categories_mask_key=gp_sources_categories_mask_key,
                     latent_key=latent_key,
                     conv_layer_encoder=conv_layer_encoder,
                     active_gp_thresh_ratio=active_gp_thresh_ratio)

In [ ]:
# Train model
model.train(n_epochs=n_epochs,
            n_epochs_all_gps=n_epochs_all_gps,
            lr=lr,
            lambda_edge_recon=lambda_edge_recon,
            lambda_gene_expr_recon=lambda_gene_expr_recon,
            lambda_l1_masked=lambda_l1_masked,
            edge_batch_size=edge_batch_size,
            n_sampled_neighbors=n_sampled_neighbors,
            use_cuda_if_available=use_cuda_if_available,
            verbose=False)

In [ ]:
# Compute latent neighbor graph
sc.pp.neighbors(model.adata,
                use_rep=latent_key,
                key_added=latent_key)

# Compute UMAP embedding
sc.tl.umap(model.adata,
           neighbors_key=latent_key)

In [ ]:
# Save trained model
model.save(dir_path=model_folder_path,
           overwrite=True,
           save_adata=True,
           adata_file_name="adata.h5ad")